# Homework - Stage 04: Data Acquisition and Ingestion

**Name:** Wang Ruihao  
**Date:** 2026-08-19

## Objectives

- Pull market data from an API with reproducible parameters.
- Load optional secrets from `.env` without exposing or committing them.
- Scrape a permitted public table with BeautifulSoup.
- Parse data types, validate the results, and save raw CSV files to `data/raw/`.

In [ ]:
# !pip install pandas requests python-dotenv beautifulsoup4 yfinance

In [1]:
# --- run me first ---
from pathlib import Path
import os, sys

if Path.cwd().name == 'notebooks':
    os.chdir('..')                       # project/notebooks -> project
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))        # allows imports from src/ later

CHECKS = [
    ('.env', 'LOCAL ONLY', 'Copy .env.example to .env; never commit it.'),
    ('.env.example', 'COMMIT', 'Safe template that belongs in the repository.'),
    ('.gitignore', 'COMMIT', 'Must contain .env.'),
]

print('Working from:', ROOT.name)
for rel, kind, note in CHECKS:
    status = 'OK' if (ROOT / rel).exists() else 'MISS'
    print(f'[{status:4}] {kind:10} {rel:14} {note}')

Working from: homework4
[OK  ] LOCAL ONLY .env           Copy .env.example to .env; never commit it.
[OK  ] COMMIT     .env.example   Safe template that belongs in the repository.
[OK  ] COMMIT     .gitignore     Must contain .env.


In [2]:
import datetime as dt
import os
import re
import subprocess
from io import StringIO

import pandas as pd

# Normal course environments use the first import. The fallback lets this
# notebook remain testable in a minimal Python installation.
try:
    import requests
except ImportError:
    from pip._vendor import requests

try:
    from bs4 import BeautifulSoup
    HAS_BS4 = True
except ImportError:
    BeautifulSoup = None
    HAS_BS4 = False

try:
    from dotenv import load_dotenv
except ImportError:
    def load_dotenv(dotenv_path=None):
        # Small fallback for simple KEY=VALUE .env files.
        path = Path(dotenv_path or '.env')
        if not path.exists():
            return False
        for raw_line in path.read_text(encoding='utf-8').splitlines():
            line = raw_line.strip()
            if not line or line.startswith('#') or '=' not in line:
                continue
            key, value = line.split('=', 1)
            os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
        return True

load_dotenv(ROOT / '.env')
RAW = ROOT / 'data' / 'raw'
RAW.mkdir(parents=True, exist_ok=True)

ALPHA_KEY = os.getenv('ALPHAVANTAGE_API_KEY', '').strip()
USER_AGENT = os.getenv(
    'DATA_USER_AGENT',
    'AFE-Stage04-Homework/1.0 (educational use)'
)

print('Raw-data folder:', RAW.relative_to(ROOT))
print('ALPHAVANTAGE_API_KEY loaded?', bool(ALPHA_KEY))

Raw-data folder: data\raw
ALPHAVANTAGE_API_KEY loaded? True


In [3]:
RUN_ID = dt.datetime.now().strftime('%Y%m%d-%H%M')

def save_csv(df: pd.DataFrame, prefix: str, **meta):
    middle = '_'.join(f'{key}-{value}' for key, value in meta.items())
    filename = f'{prefix}_{middle}_{RUN_ID}.csv' if middle else f'{prefix}_{RUN_ID}.csv'
    path = RAW / filename
    df.to_csv(path, index=False)
    print(f'Saved {len(df):,} rows to {path.relative_to(ROOT)}')
    return path


def validate(df: pd.DataFrame, required, numeric=None, text=None, unique=None):
    numeric = numeric or []
    text = text or []
    unique = unique or []

    missing = [column for column in required if column not in df.columns]
    numeric_invalid = {
        column: int((df[column].notna() & pd.to_numeric(df[column], errors='coerce').isna()).sum())
        for column in numeric if column in df.columns
    }
    empty_text = {
        column: int(df[column].fillna('').astype(str).str.strip().eq('').sum())
        for column in text if column in df.columns
    }
    duplicate_rows = int(df.duplicated(subset=unique).sum()) if unique and not missing else 0

    report = {
        'missing_columns': missing,
        'shape': tuple(df.shape),
        'na_by_column': {column: int(value) for column, value in df.isna().sum().items()},
        'na_total': int(df.isna().sum().sum()),
        'numeric_invalid': numeric_invalid,
        'empty_text': empty_text,
        'duplicate_keys': duplicate_rows,
    }
    report['passed'] = (
        not missing
        and len(df) > 0
        and all(value == 0 for value in numeric_invalid.values())
        and all(value == 0 for value in empty_text.values())
        and duplicate_rows == 0
    )
    return report

## Part 1 - API pull (required)

The notebook first tries Alpha Vantage when `ALPHAVANTAGE_API_KEY` exists in `.env`. If no key is present, or the Alpha Vantage request fails, it uses the Yahoo Finance chart endpoint through `requests`. Both routes produce the same two-column schema: `date` and `adj_close`.

In [4]:
SYMBOL = 'AAPL'
RANGE = '3mo'
INTERVAL = '1d'

ALPHA_URL = 'https://www.alphavantage.co/query'
YAHOO_URL = f'https://query1.finance.yahoo.com/v8/finance/chart/{SYMBOL}'


def fetch_alpha_vantage(symbol, api_key):
    params = {
        'function': 'TIME_SERIES_DAILY_ADJUSTED',
        'symbol': symbol,
        'outputsize': 'compact',
        'apikey': api_key,
    }
    response = requests.get(ALPHA_URL, params=params, headers={'User-Agent': USER_AGENT}, timeout=30)
    response.raise_for_status()
    payload = response.json()
    series_key = next((key for key in payload if 'Time Series' in key), None)
    if series_key is None:
        message = payload.get('Note') or payload.get('Information') or payload.get('Error Message')
        raise ValueError(message or 'Alpha Vantage response did not contain a time series.')

    frame = pd.DataFrame.from_dict(payload[series_key], orient='index')
    price_column = '5. adjusted close' if '5. adjusted close' in frame.columns else '4. close'
    frame = frame.reset_index().rename(columns={'index': 'date', price_column: 'adj_close'})
    return frame[['date', 'adj_close']], 'alphavantage'


def fetch_yahoo_chart(symbol, range_value='3mo', interval='1d'):
    params = {
        'range': range_value,
        'interval': interval,
        'events': 'div,splits',
        'includeAdjustedClose': 'true',
    }
    response = requests.get(YAHOO_URL, params=params, headers={'User-Agent': USER_AGENT}, timeout=30)
    response.raise_for_status()
    payload = response.json()
    results = payload.get('chart', {}).get('result') or []
    if not results:
        error = payload.get('chart', {}).get('error')
        raise ValueError(f'Yahoo response did not contain chart data: {error}')

    result = results[0]
    timestamps = result.get('timestamp', [])
    quote = result.get('indicators', {}).get('quote', [{}])[0]
    adjusted = result.get('indicators', {}).get('adjclose', [{}])[0].get('adjclose')
    prices = adjusted if adjusted is not None else quote.get('close', [])
    if len(timestamps) != len(prices):
        raise ValueError('Yahoo timestamp and price arrays have different lengths.')

    frame = pd.DataFrame({'date': timestamps, 'adj_close': prices})
    frame['date'] = pd.to_datetime(frame['date'], unit='s', utc=True).dt.tz_convert('America/New_York').dt.tz_localize(None)
    return frame, 'yahoo'


if ALPHA_KEY:
    try:
        df_api, API_SOURCE = fetch_alpha_vantage(SYMBOL, ALPHA_KEY)
    except Exception as exc:
        print(f'Alpha Vantage failed ({type(exc).__name__}); using Yahoo fallback.')
        df_api, API_SOURCE = fetch_yahoo_chart(SYMBOL, RANGE, INTERVAL)
else:
    df_api, API_SOURCE = fetch_yahoo_chart(SYMBOL, RANGE, INTERVAL)

df_api['date'] = pd.to_datetime(df_api['date'], errors='coerce')
df_api['adj_close'] = pd.to_numeric(df_api['adj_close'], errors='coerce')
df_api = df_api.dropna(subset=['date', 'adj_close']).sort_values('date').reset_index(drop=True)

print(f'Source: {API_SOURCE}; symbol: {SYMBOL}; rows: {len(df_api):,}')
print(f'Date range: {df_api.date.min().date()} to {df_api.date.max().date()}')
df_api.head()

Alpha Vantage failed (ValueError); using Yahoo fallback.
Source: yahoo; symbol: AAPL; rows: 64
Date range: 2026-05-18 to 2026-08-18


,date,adj_close
0,2026-05-18 09:30:00,297.583344
1,2026-05-19 09:30:00,298.712372
2,2026-05-20 09:30:00,301.989563
3,2026-05-21 09:30:00,304.727173
4,2026-05-22 09:30:00,308.553894


In [5]:
v_api = validate(
    df_api,
    required=['date', 'adj_close'],
    numeric=['adj_close'],
    unique=['date'],
)
v_api['date_is_datetime'] = bool(pd.api.types.is_datetime64_any_dtype(df_api['date']))
v_api['price_is_numeric'] = bool(pd.api.types.is_numeric_dtype(df_api['adj_close']))
v_api['passed'] = v_api['passed'] and v_api['date_is_datetime'] and v_api['price_is_numeric']

assert v_api['passed'], v_api
v_api

{'missing_columns': [],
 'shape': (64, 2),
 'na_by_column': {'date': 0, 'adj_close': 0},
 'na_total': 0,
 'numeric_invalid': {'adj_close': 0},
 'empty_text': {},
 'duplicate_keys': 0,
 'passed': True,
 'date_is_datetime': True,
 'price_is_numeric': True}

In [8]:
api_path = save_csv(
    df_api,
    prefix='api',
    source=API_SOURCE,
    symbol=SYMBOL,
)
pd.read_csv(api_path).head(3)

Saved 64 rows to data\raw\api_source-yahoo_symbol-AAPL_20260819-0258.csv


,date,adj_close
0,2026-05-18 09:30:00,297.583344
1,2026-05-19 09:30:00,298.712372
2,2026-05-20 09:30:00,301.989563


## Part 2 - Scrape a public table (optional)

The source is Wikipedia's public **List of S&P 500 companies** page. 

In [9]:
SCRAPE_URL = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': USER_AGENT}

response = requests.get(SCRAPE_URL, headers=headers, timeout=30)
response.raise_for_status()

if HAS_BS4:
    soup = BeautifulSoup(response.text, 'html.parser')
    table = soup.select_one('table#constituents')

    if table is None:
        # Resilient fallback: choose a table whose header contains both fields.
        for candidate in soup.find_all('table'):
            header_text = ' '.join(cell.get_text(' ', strip=True) for cell in candidate.find_all('th'))
            if 'Symbol' in header_text and 'Security' in header_text:
                table = candidate
                break
    if table is None:
        raise ValueError('Could not find the S&P 500 constituents table.')

    parsed_rows = [
        [cell.get_text(' ', strip=True) for cell in row.find_all(['th', 'td'])]
        for row in table.find_all('tr')
    ]
    parsed_rows = [row for row in parsed_rows if row]
    header, *data = parsed_rows
    data = [row for row in data if len(row) == len(header)]
    df_scrape = pd.DataFrame(data, columns=header)
else:
    # Execution-only fallback for minimal environments without beautifulsoup4.
    # The normal course environment uses the BeautifulSoup branch above.
    df_scrape = pd.read_html(StringIO(response.text), attrs={'id': 'constituents'})[0]

df_scrape.columns = [re.sub(r'\[.*?\]', '', str(column)).strip() for column in df_scrape.columns]
df_scrape = df_scrape.rename(columns={
    'Symbol': 'ticker',
    'Security': 'company',
    'GICS Sector': 'sector',
    'GICS Sub-Industry': 'sub_industry',
    'Headquarters Location': 'headquarters',
    'Date added': 'date_added',
    'CIK': 'cik',
})

selected_columns = [
    'ticker', 'company', 'sector', 'sub_industry',
    'headquarters', 'date_added', 'cik'
]
missing_selected = [column for column in selected_columns if column not in df_scrape.columns]
if missing_selected:
    raise ValueError(f'Scraped table schema changed; missing {missing_selected}')

df_scrape = df_scrape[selected_columns].copy()
df_scrape['ticker'] = df_scrape['ticker'].astype('string').str.replace(r'\[.*?\]', '', regex=True).str.strip()
for column in ['company', 'sector', 'sub_industry', 'headquarters']:
    df_scrape[column] = df_scrape[column].astype('string').str.strip()
df_scrape['date_added'] = pd.to_datetime(df_scrape['date_added'], errors='coerce')
df_scrape['cik'] = pd.to_numeric(df_scrape['cik'], errors='coerce').astype('Int64')

print(f'Scraped {len(df_scrape):,} rows and {len(df_scrape.columns)} columns.')
df_scrape.head()

Scraped 503 rows and 7 columns.


,ticker,company,sector,sub_industry,headquarters,date_added,cik
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee , Wisconsin",2017-07-26,91142
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin , Ireland",2011-07-06,1467373


In [10]:
v_scrape = validate(
    df_scrape,
    required=['ticker', 'company', 'sector', 'date_added', 'cik'],
    numeric=['cik'],
    text=['ticker', 'company', 'sector'],
    unique=['ticker'],
)
v_scrape['date_is_datetime'] = bool(pd.api.types.is_datetime64_any_dtype(df_scrape['date_added']))
v_scrape['date_parse_failures'] = int(df_scrape['date_added'].isna().sum())
v_scrape['passed'] = v_scrape['passed'] and v_scrape['date_is_datetime'] and v_scrape['date_parse_failures'] == 0

assert v_scrape['passed'], v_scrape
v_scrape

{'missing_columns': [],
 'shape': (503, 7),
 'na_by_column': {'ticker': 0,
  'company': 0,
  'sector': 0,
  'sub_industry': 0,
  'headquarters': 0,
  'date_added': 0,
  'cik': 1},
 'na_total': 1,
 'numeric_invalid': {'cik': 0},
 'empty_text': {'ticker': 0, 'company': 0, 'sector': 0},
 'duplicate_keys': 0,
 'passed': True,
 'date_is_datetime': True,
 'date_parse_failures': 0}

In [11]:
scrape_path = save_csv(
    df_scrape,
    prefix='scrape',
    site='wikipedia',
    table='sp500-constituents',
)
pd.read_csv(scrape_path).head(3)

Saved 503 rows to data\raw\scrape_site-wikipedia_table-sp500-constituents_20260819-0258.csv


,ticker,company,sector,sub_industry,headquarters,date_added,cik
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740.0
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee , Wisconsin",2017-07-26,91142.0
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800.0
